# 📚 ДЗ №2: Работа с данными для LLM

## 🎯 Цель задания
После выполнения задания вы сможете:
- Предобрабатывать русскоязычные текстовые данные для LLM
- Работать с готовыми моделями HuggingFace для анализа тональности и NER
- Создавать эффективные промпты для LLM API
- Сравнивать качество работы разных подходов к анализу текста
- Формировать датасеты в формате instruction-following для fine-tuning
- Сохранять данные в правильных форматах для обучения LLM

## 📝 Структура задания
- **Часть 1** (35% оценки): Предобработка данных и работа с готовыми моделями
- **Часть 2** (35% оценки): LLM API и prompt engineering
- **Часть 3** (20% оценки): Подготовка данных для fine-tuning LLM
- **Часть 4** (10% оценки): Сравнительный анализ и визуализация

## ⚡ Критерии оценки
- Качество предобработки данных: 25%
- Корректность работы с готовыми моделями: 20%
- Эффективность промптов для LLM: 25%
- Правильность подготовки данных для fine-tuning: 20%
- Качество сравнительного анализа: 10%


## 🔧 Установка зависимостей

Установим необходимые библиотеки для работы с данными, готовыми моделями и LLM API.


In [1]:
%pip install pandas numpy matplotlib seaborn
%pip install transformers torch
%pip install openai>=1.0.0  # Для работы с OpenAI API
%pip install datasets
%pip install pymorphy2



Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
zsh:1: 1.0.0 not found
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.2/34.2 MB 4.7 MB/s  0:00:07m0:00:0100:01
  Attempting uninstall: fsspec━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/14 [pyarrow]
    Found existing installation: fsspec 2026.1.0━━━━━━━━━━━━━━  1/14 [pyarrow]
    Uninstalling fsspec-2026.1.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/14 [pyarrow]
      Successfully uninstalled fsspec-2026.1.0━━━━━━━━━━━━━━━━  1/14 [pyarrow]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [datasets]/14 [datasets]ess]
Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 8.0 MB/s  0:00:01m0:00:01:00:01
  Cr

In [11]:
# Импорт необходимых библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from typing import List, Dict, Tuple
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification
import warnings
warnings.filterwarnings('ignore')

# Настройка отображения
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

print("Библиотеки загружены успешно!")



Библиотеки загружены успешно!


## 📊 Часть 1: Предобработка данных и готовые модели (35% оценки)

### Задание 1.1: Анализ "грязного" датасета

Проанализируем реалистичный датасет с типичными проблемами: опечатки, разные регистры, лишние пробелы, эмодзи.


In [36]:
# Создаем "грязный" датасет с типичными проблемами реальных данных
# Включаем сложные случаи для демонстрации преимуществ LLM
raw_reviews = [
    # Простые случаи
    "отличный iphone 14 PRO!!!  купил в магазине  apple на тверской 😊. Камера супер",
    "УЖАСНОЕ обслуживание в сбербанке на красной площади.. менеджер иван петров вобще не помог(",

    # Сарказм и ирония (сложно для классических моделей)
    "Спасибо огромное сотрудникам МТС за то что 3 часа держали меня в очереди! Просто восхитительно 👏",
    "Какой замечательный сервис в Пятерочке - касса сломалась прямо передо мной, а персонал даже не извинился",

    # Смешанные эмоции
    "iPhone 13 хороший телефон, но цена кусается. В целом доволен покупкой в re:Store",
    "Ресторан Белуга красивый и атмосфера приятная, но официант Максим был невнимателен",

    # Сложная структура предложений
    "Хотя Tesla Model Y и дорогая машина, и сервис в Рольф Премиум иногда подводит, но в целом я очень доволен покупкой",
    "Не могу сказать что отель Ритц-Карлтон плохой, просто ожидал большего за такие деньги",

    # Контекстно-зависимые случаи
    "Заказал доставку в Яндекс.Еде из ресторана Дача на Рублевке - привезли холодное, но курьер Андрей был вежливый",
    "MacBook Pro 16 работает как часы уже год, покупал в iStore на Арбате у консультанта Елены",

    # Неоднозначные случаи
    "Сходил в кинотеатр Октябрь посмотреть новый фильм Marvel - ну такое себе, но попкорн вкусный был",
    "Обслуживание в банке ВТБ на Тверской оставляет желать лучшего, хотя менеджер Ольга старалась помочь",

    # Сложные именованные сущности
    "Купил новый Samsung Galaxy S24 Ultra в DNS на Ленинском проспекте, консультант Дмитрий Иванович всё объяснил",
    "Ужинал в ресторане White Rabbit на Смоленской площади - шеф-повар Владимир Мухин превзошел ожидания",

    # Опечатки и сленг
    "норм телек LG купил в эльдорадо, продавец норм чел был, всё рассказал про функции"
]

# TODO: Создайте DataFrame и проанализируйте проблемы в данных
# Создайте DataFrame из списка raw_reviews
# Добавьте колонку с правильными метками тональности для каждого отзыва
# Проанализируйте и выведите список проблем, которые вы видите в данных
# Подумайте: какие проблемы могут повлиять на качество анализа?

# Ваш код здесь:
sentiment_labels = [
    'POSITIVE',   # "отличный iphone 14 PRO!!!"
    'NEGATIVE',   # "УЖАСНОЕ обслуживание в сбербанке"
    'NEGATIVE',   # "Спасибо огромное... 3 часа держали" (сарказм)
    'NEGATIVE',   # "Какой замечательный сервис" (ирония)
    'NEUTRAL',    # "iPhone 13 хороший... но цена кусается"
    'NEUTRAL',    # "Ресторан Белуга красивый... но официант"
    'POSITIVE',   # "в целом я очень доволен покупкой"
    'NEUTRAL',    # "Не могу сказать что... плохой"
    'NEUTRAL',    # "привезли холодное, но курьер... вежливый"
    'POSITIVE',   # "MacBook Pro 16 работает как часы"
    'NEUTRAL',    # "ну такое себе, но попкорн вкусный"
    'NEGATIVE',   # "оставляет желать лучшего"
    'POSITIVE',   # "Samsung Galaxy S24 Ultra... всё объяснил"
    'POSITIVE',   # "превзошел ожидания"
    'POSITIVE'    # "норм телек... норм чел был"
]
df = pd.DataFrame(raw_reviews, columns=['review'])
df['sentiment'] = sentiment_labels
print(sentiment_labels)
print(df.head())


['POSITIVE', 'NEGATIVE', 'NEGATIVE', 'NEGATIVE', 'NEUTRAL', 'NEUTRAL', 'POSITIVE', 'NEUTRAL', 'NEUTRAL', 'POSITIVE', 'NEUTRAL', 'NEGATIVE', 'POSITIVE', 'POSITIVE', 'POSITIVE']
                                                                                                     review  \
0                            отличный iphone 14 PRO!!!  купил в магазине  apple на тверской 😊. Камера супер   
1                УЖАСНОЕ обслуживание в сбербанке на красной площади.. менеджер иван петров вобще не помог(   
2          Спасибо огромное сотрудникам МТС за то что 3 часа держали меня в очереди! Просто восхитительно 👏   
3  Какой замечательный сервис в Пятерочке - касса сломалась прямо передо мной, а персонал даже не извинился   
4                          iPhone 13 хороший телефон, но цена кусается. В целом доволен покупкой в re:Store   

  sentiment  
0  POSITIVE  
1  NEGATIVE  
2  NEGATIVE  
3  NEGATIVE  
4   NEUTRAL  


In [13]:
print("=" * 60)
print("АНАЛИЗ ДАТАСЕТА")
print("=" * 60)

print(f"\n📊 Общая информация:")
print(f"   Количество отзывов: {len(df)}")
print(f"   Колонки: {df.columns.tolist()}")

print(f"\n⚠️  Выявленные проблемы в данных:")

# 1. Проверка на разный регистр
has_uppercase = df['review'].str.contains('[A-Z]{2,}').sum()
print(f"   1. Тексты с ЗАГЛАВНЫМИ буквами: {has_uppercase}")

# 2. Проверка на эмодзи
has_emoji = df['review'].str.contains('😊|👏|😢|😁|🙂', regex=True).sum()
print(f"   2. Тексты с эмодзи: {has_emoji}")

# 3. Проверка на множественные пробелы
has_extra_spaces = df['review'].str.contains('\s{2,}', regex=True).sum()
print(f"   3. Тексты с лишними пробелами: {has_extra_spaces}")

# 4. Проверка на повторяющуюся пунктуацию
has_repeated_punct = df['review'].str.contains('[.!?]{2,}', regex=True).sum()
print(f"   4. Тексты с повторяющейся пунктуацией: {has_repeated_punct}")

# 5. Проверка на сленг
has_slang = df['review'].str.contains('норм|чел|телек', case=False).sum()
print(f"   5. Тексты со сленгом: {has_slang}")

# 6. Распределение по тональности
print(f"\n📈 Распределение тональности:")
print(df['sentiment'].value_counts())

АНАЛИЗ ДАТАСЕТА

📊 Общая информация:
   Количество отзывов: 15
   Колонки: ['review', 'sentiment']

⚠️  Выявленные проблемы в данных:
   1. Тексты с ЗАГЛАВНЫМИ буквами: 3
   2. Тексты с эмодзи: 2
   3. Тексты с лишними пробелами: 1
   4. Тексты с повторяющейся пунктуацией: 2
   5. Тексты со сленгом: 1

📈 Распределение тональности:
sentiment
POSITIVE    6
NEUTRAL     5
NEGATIVE    4
Name: count, dtype: int64


### Задание 1.2: Очистка и нормализация данных


In [37]:
def clean_text(text: str) -> str:
    """
    Очистка и нормализация русскоязычного текста
    """
    # TODO: Реализуйте базовую очистку текста
    # Подумайте над следующими аспектами:
    # - Как убрать эмодзи и специальные символы?
    # - Как нормализовать пробелы и отступы?
    # - Нужно ли исправлять регистр? Как?
    # - Что делать с повторяющимися знаками препинания?
    # - Как разделить слитно написанные слова (например, iPhone14)?

    # Используйте регулярные выражения (модуль re)
    # Ваш код здесь: 

    # Удаляем эмодзи
    text = re.sub(r'[^\w\s\.\,\!\?\-]', '', text)
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # эмотиконы
        u"\U0001F300-\U0001F5FF"  # символы
        u"\U0001F680-\U0001F6FF"  # транспорт
        u"\U0001F1E0-\U0001F1FF"  # флаги
        "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r'', text)

    # Нормализуем пробелы
    text = re.sub(r'\s+', ' ', text).strip()

    # Убираем повторы пунктуации
    text = re.sub(r'([.!?]){2,}', r'\1', text)

    # Разделяем слитные слова (iPhone14 → iPhone 14)
    text = re.sub(r'([a-zA-Zа-яА-Я]+)(\d+)', r'\1 \2', text)

    # Весь текст приводить к нижнему регистру может быть плохо - важно сохранять названия брендов, в том числе для NER
    # Для определения тональности можно весь текст приводить к нижнему регистру, хотя при этом возможно потеряется эмоциональная тональность 
    # (можно управлять этим через параметр функции)
    # Список известных аббревиатур
    abbrevs = {'МТС', 'ВТБ', 'DNS', 'SMS', 'USA', 'CEO', 'LG'}
    
    def fix_all_caps(match):
        word = match.group(0)
        # Сохраняем известные аббревиатуры
        if word in abbrevs:
            return word
        # Для остальных: capitalize либо lower
        return word.lower()
    
    text = re.sub(r'\b[А-ЯA-Z]{2,}\b', fix_all_caps, text)

    return text


# TODO: Примените функцию очистки к данным и сравните результаты
# Создайте новую колонку с очищенными текстами
# Сравните исходные и очищенные тексты
df['text_cleaned'] = df['review'].apply(clean_text)
df['text_lowercase'] = df['text_cleaned'].str.lower()

pd.set_option('display.max_colwidth', 60)
display(df[['review', 'text_cleaned']].sample(5))
df[['review', 'text_cleaned']].apply(lambda x: x.str.len()).describe()

,review,text_cleaned
1,УЖАСНОЕ обслуживание в сбербанке на красной площади.. ме...,ужасное обслуживание в сбербанке на красной площади. мен...
5,"Ресторан Белуга красивый и атмосфера приятная, но официа...","Ресторан Белуга красивый и атмосфера приятная, но официа..."
7,"Не могу сказать что отель Ритц-Карлтон плохой, просто ож...","Не могу сказать что отель Ритц-Карлтон плохой, просто ож..."
8,Заказал доставку в Яндекс.Еде из ресторана Дача на Рубле...,Заказал доставку в Яндекс.Еде из ресторана Дача на Рубле...
4,"iPhone 13 хороший телефон, но цена кусается. В целом дов...","iPhone 13 хороший телефон, но цена кусается. В целом дов..."


,review,text_cleaned
count,15.000000,15.000000
mean,94.066667,93.466667
std,11.640242,12.397388
min,78.000000,73.000000
25%,83.500000,83.500000
50%,96.000000,94.000000
75%,101.500000,101.500000
max,114.000000,114.000000


### Задание 1.3: Использование готовых моделей HuggingFace


In [38]:
%pip install python-dotenv
from dotenv import load_dotenv

load_dotenv()

# TODO: Загрузите готовые модели HuggingFace для анализа тональности и NER
# Исследуйте HuggingFace Hub и найдите подходящие русскоязычные модели для:
# - Анализа тональности (sentiment analysis)
# - Извлечения именованных сущностей (NER)
#
# Используйте функцию pipeline() из библиотеки transformers
# Обратите внимание на параметры модели и токенизатора

# Ваш код для загрузки моделей:
print("Загрузка модели для анализа тональности...")
sentiment_pipeline = pipeline(
    "text-classification",
    model="blanchefort/rubert-base-cased-sentiment",
    truncation=True,
    max_length=512
)

print("Загрузка модели для NER...")
ner_pipeline = pipeline(
    "ner",
    # model="Gherman/bert-base-NER-Russian",
    model="julian-schelb/roberta-ner-multilingual",
    aggregation_strategy="simple",  # Объединяет токены в целые сущности
)



Note: you may need to restart the kernel to use updated packages.
Загрузка модели для анализа тональности...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: blanchefort/rubert-base-cased-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Загрузка модели для NER...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: julian-schelb/roberta-ner-multilingual
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
def analyze_with_huggingface(texts: List[str]) -> List[Dict]:
    """
    Анализ текстов с помощью готовых моделей HuggingFace
    """
    # TODO: Реализуйте функцию анализа
    # Для каждого текста:
    # 1. Примените модель анализа тональности
    # 2. Примените модель NER
    # 3. Соберите результаты в структурированном виде
    # 4. Верните список словарей с результатами
    results = []

    for i, text in enumerate(texts):
        print(f"Анализ текста {i+1}/{len(texts)}...")

        sentiment_result = sentiment_pipeline(text)[0]
        ner_result = ner_pipeline(text)

        result = {
            'text': text,
            'sentiment': {
                'label': sentiment_result['label'],
                'score': round(sentiment_result['score'], 4)
            },
            'entities': []
        }

        for entity in ner_result:
            result['entities'].append({
                'text': entity['word'],
                'type': entity['entity_group'],
                'score': round(entity['score'], 4)
            })
    
        results.append(result)

    return results



In [39]:
from pprint import pprint
# TODO: Протестируйте модели на очищенных данных
# Проанализируйте несколько текстов (для начала возьмите 3-5)
# Выведите результаты в понятном формате
# Проанализируйте качество работы моделей
test_indices = [0, 1, 4, 6, 14]
test_texts = df.iloc[test_indices]['text_cleaned'].tolist()

hf_test_results = analyze_with_huggingface(test_texts)
pprint(hf_test_results)



Анализ текста 1/5...
Анализ текста 2/5...
Анализ текста 3/5...
Анализ текста 4/5...
Анализ текста 5/5...
[{'entities': [{'score': np.float32(0.9532), 'text': 'apple', 'type': 'ORG'},
               {'score': np.float32(0.7863), 'text': '', 'type': 'LOC'},
               {'score': np.float32(0.7984), 'text': 'твер', 'type': 'LOC'},
               {'score': np.float32(0.7546), 'text': 'ской', 'type': 'LOC'}],
  'sentiment': {'label': 'POSITIVE', 'score': 0.9366},
  'text': 'отличный iphone 14 pro! купил в магазине apple на тверской . Камера '
          'супер'},
 {'entities': [{'score': np.float32(0.7314), 'text': 'с', 'type': 'ORG'},
               {'score': np.float32(0.7442), 'text': 'бер', 'type': 'ORG'},
               {'score': np.float32(0.8527), 'text': 'банк', 'type': 'ORG'},
               {'score': np.float32(0.8438), 'text': 'е', 'type': 'ORG'},
               {'score': np.float32(0.9953), 'text': 'и', 'type': 'PER'},
               {'score': np.float32(0.9935),
             

In [18]:
hf_results = analyze_with_huggingface(df['text_cleaned'].tolist())
predictions = [result['sentiment']['label'] for result in hf_results]
df['hf_sentiment_predicted'] = predictions
df['hf_is_correct'] = df['sentiment'] == df['hf_sentiment_predicted']
accuracy = df['hf_is_correct'].sum() / len(df)
print(f"Точность определения тональности: {accuracy:.1%}")
for i, row in df.iterrows():
    status = "✓" if row['hf_is_correct'] else "✗"
    print(f"\n{i+1}. {status} Текст: {row['text_cleaned'][:60]}...")
    print(f"   Истинная метка: {row['sentiment']:8} | Предсказание: {row['hf_sentiment_predicted']:8}")


Анализ текста 1/15...
Анализ текста 2/15...
Анализ текста 3/15...
Анализ текста 4/15...
Анализ текста 5/15...
Анализ текста 6/15...
Анализ текста 7/15...
Анализ текста 8/15...
Анализ текста 9/15...
Анализ текста 10/15...
Анализ текста 11/15...
Анализ текста 12/15...
Анализ текста 13/15...
Анализ текста 14/15...
Анализ текста 15/15...
Точность определения тональности: 46.7%

1. ✓ Текст: отличный iphone 14 pro! купил в магазине apple на тверской ....
   Истинная метка: POSITIVE | Предсказание: POSITIVE

2. ✓ Текст: ужасное обслуживание в сбербанке на красной площади. менедже...
   Истинная метка: NEGATIVE | Предсказание: NEGATIVE

3. ✗ Текст: Спасибо огромное сотрудникам МТС за то что 3 часа держали ме...
   Истинная метка: NEGATIVE | Предсказание: POSITIVE

4. ✓ Текст: Какой замечательный сервис в Пятерочке - касса сломалась пря...
   Истинная метка: NEGATIVE | Предсказание: NEGATIVE

5. ✗ Текст: iPhone 13 хороший телефон, но цена кусается. В целом доволен...
   Истинная метка: NEUTRAL 

In [40]:
for i, result in enumerate(hf_results):
    if result['entities']:
        print(f"\n{i+1}. Текст: {result['text'][:60]}...")
        print(f"   Найдено сущностей: {len(result['entities'])}")
        for entity in result['entities']:
            print(f"   - [{entity['type']}] {entity['text']} (score: {entity['score']:.3f})")
    else:
        print(f"\n{i+1}. Текст: {result['text'][:60]}...")
        print(f"   Сущностей не найдено")


1. Текст: отличный iphone 14 pro! купил в магазине apple на тверской ....
   Найдено сущностей: 4
   - [ORG] apple (score: 0.953)
   - [LOC]  (score: 0.786)
   - [LOC] твер (score: 0.798)
   - [LOC] ской (score: 0.755)

2. Текст: ужасное обслуживание в сбербанке на красной площади. менедже...
   Найдено сущностей: 6
   - [ORG] с (score: 0.731)
   - [ORG] бер (score: 0.744)
   - [ORG] банк (score: 0.853)
   - [ORG] е (score: 0.844)
   - [PER] и (score: 0.995)
   - [PER] ван петров (score: 0.993)

3. Текст: Спасибо огромное сотрудникам МТС за то что 3 часа держали ме...
   Найдено сущностей: 2
   - [ORG] М (score: 0.997)
   - [ORG] ТС (score: 0.998)

4. Текст: Какой замечательный сервис в Пятерочке - касса сломалась пря...
   Найдено сущностей: 3
   - [LOC] Пят (score: 0.591)
   - [LOC] еро (score: 0.567)
   - [LOC] чке (score: 0.596)

5. Текст: iPhone 13 хороший телефон, но цена кусается. В целом доволен...
   Найдено сущностей: 1
   - [ORG] iPhone 13 (score: 0.824)

6. Текст: Ресторан

Анализ качества определения тональности:
- Определение тональности хорошо срабатывает только в явно положительных или отрицательных случаях. И путается, когда в предложении есть и позитивные и негативные окраски, а также в случаях иронии/сарказма

Анализ качества NER
- Русскоязычные модели (Gherman) не распознают организации, но хорошо определяют имена, фамилии людей
- Мультиязычные модели (RoBERTa) плохо токенизируют кириллицу, зато англоязычные организации выделяют неплохо (хотя LG пропустил)
- Выделить модели оборудования не смогли и мультиязычные модели
- Не все модели доступны на HF и некоторые надо ставить отдельными пакетами
- Модели не универсальны и надо использовать под каждую задачу (персоны, организации, даты) свою модель, не забывая о том какие в них используются токенизаторы

## 🤖 Часть 2: LLM API и Prompt Engineering (35% оценки)

### Задание 2.1: Создание эффективных промптов


In [60]:
def create_prompts_for_llm() -> Dict[str, str]:
    """
    Создание базовых промптов для разных задач (один промпт на задачу)
    """
    # TODO: Создайте эффективные промпты для NER и sentiment analysis
    # Подумайте о структуре хорошего промпта:
    # - Четкое описание задачи
    # - Примеры входных и выходных данных
    # - Формат ответа (JSON, текст и т.д.)
    # - Особые требования (например, для русского языка)

    # Создайте промпты для:
    # 1. Извлечения именованных сущностей (NER)
    # 2. Анализа тональности (sentiment analysis)

    # Ваш код здесь:
    prompts = {
        "sentiment_analysis": """Ты - эксперт по анализу тональности русскоязычных текстов.

Твоя задача: определить эмоциональную окраску отзыва.

ВАЖНО:
- Учитывай сарказм и иронию (положительные слова могут означать негатив)
- Различай смешанные эмоции
- Анализируй контекст, а не только отдельные слова

КЛАССЫ ТОНАЛЬНОСТИ:
- POSITIVE: явно позитивный отзыв, клиент доволен
- NEUTRAL: смешанные эмоции или нейтральная оценка
- NEGATIVE: негативный отзыв, недовольство, жалоба

ФОРМАТ ОТВЕТА - только JSON:
{{
  "sentiment": "POSITIVE/NEUTRAL/NEGATIVE",
  "confidence": 0.0-1.0,
  "reasoning": "краткое объяснение"
}}

ПРИМЕРЫ:

Текст: "Отличный телефон, камера супер!"
Ответ: {{"sentiment": "POSITIVE", "confidence": 0.95, "reasoning": "Явные положительные оценки"}}

Текст: "Спасибо огромное за 3 часа ожидания в очереди!"
Ответ: {{"sentiment": "NEGATIVE", "confidence": 0.9, "reasoning": "Сарказм - благодарность за плохое обслуживание"}}

Текст: "Телефон хороший, но цена высокая"
Ответ: {{"sentiment": "NEUTRAL", "confidence": 0.8, "reasoning": "Смешанные эмоции: плюсы и минусы"}}

АНАЛИЗИРУЙ ТЕКСТ:
{text}""",

        "ner": """Ты - эксперт по извлечению именованных сущностей из русскоязычных текстов.

Твоя задача: найти и классифицировать все упоминания сущностей в тексте.

ТИПЫ СУЩНОСТЕЙ:
- PERSON: имена людей (Иван, Елена Петрова, Владимир Мухин)
- ORGANIZATION: компании, бренды, организации (Apple, Сбербанк, МТС, ВТБ)
- LOCATION: места (Москва, Тверская, Красная площадь, Арбат)
- PRODUCT: товары и модели (iPhone 14 Pro, Tesla Model Y, MacBook Pro 16)
- DATE: даты (15 января, вчера, 2024 год)
- MONEY: денежные суммы (1000 рублей, $50)

ВАЖНО:
- Извлекай полные названия (не "iPhone", а "iPhone 14 Pro")
- Различай бренды (ORGANIZATION) и продукты (PRODUCT)
- Имена + фамилии объединяй в одну сущность

ФОРМАТ ОТВЕТА - только JSON:
{{
  "entities": [
    {{"text": "найденный текст", "type": "тип", "start": позиция, "end": позиция}},
    ...
  ]
}}

ПРИМЕРЫ:

Текст: "Иван Петров купил iPhone 14 в магазине Apple на Тверской"
Ответ: {{
  "entities": [
    {{"text": "Иван Петров", "type": "PERSON", "start": 0, "end": 11}},
    {{"text": "iPhone 14", "type": "PRODUCT", "start": 19, "end": 28}},
    {{"text": "Apple", "type": "ORGANIZATION", "start": 41, "end": 46}},
    {{"text": "Тверской", "type": "LOCATION", "start": 50, "end": 58}}
  ]
}}

Текст: "Обслуживание в банке ВТБ оставляет желать лучшего"
Ответ: {{
  "entities": [
    {{"text": "ВТБ", "type": "ORGANIZATION", "start": 21, "end": 24}}
  ]
}}

АНАЛИЗИРУЙ ТЕКСТ:
{text}""",
    }

    return prompts

# TODO: Протестируйте ваши промпты
# Выведите созданные промпты и оцените их качество
prompts = create_prompts_for_llm()
print("\nПРОМПТ ДЛЯ SENTIMENT ANALYSIS:")
print(prompts["sentiment_analysis"])
print("\nПРОМПТ ДЛЯ NER:")
print(prompts["ner"])





ПРОМПТ ДЛЯ SENTIMENT ANALYSIS:
Ты - эксперт по анализу тональности русскоязычных текстов.

Твоя задача: определить эмоциональную окраску отзыва.

ВАЖНО:
- Учитывай сарказм и иронию (положительные слова могут означать негатив)
- Различай смешанные эмоции
- Анализируй контекст, а не только отдельные слова

КЛАССЫ ТОНАЛЬНОСТИ:
- POSITIVE: явно позитивный отзыв, клиент доволен
- NEUTRAL: смешанные эмоции или нейтральная оценка
- NEGATIVE: негативный отзыв, недовольство, жалоба

ФОРМАТ ОТВЕТА - только JSON:
{{
  "sentiment": "POSITIVE/NEUTRAL/NEGATIVE",
  "confidence": 0.0-1.0,
  "reasoning": "краткое объяснение"
}}

ПРИМЕРЫ:

Текст: "Отличный телефон, камера супер!"
Ответ: {{"sentiment": "POSITIVE", "confidence": 0.95, "reasoning": "Явные положительные оценки"}}

Текст: "Спасибо огромное за 3 часа ожидания в очереди!"
Ответ: {{"sentiment": "NEGATIVE", "confidence": 0.9, "reasoning": "Сарказм - благодарность за плохое обслуживание"}}

Текст: "Телефон хороший, но цена высокая"
Ответ: {{"sen

In [42]:
# TODO: Настройте OpenAI API
# Установите API ключ через переменные окружения
# Изучите документацию OpenAI API для Python
from dotenv import load_dotenv
from openai import OpenAI
import os
import httpx

if 'OPENAI_BASE_URL' in os.environ:
    del os.environ['OPENAI_BASE_URL']
if 'OPENAI_API_KEY' in os.environ:
    del os.environ['OPENAI_API_KEY']

# Перезагружаем .env с override=True
load_dotenv(override=True)

http_client = httpx.Client(
    verify=False,  # Отключаем проверку SSL сертификата
    timeout=60.0
)

client = OpenAI(
    api_key=os.getenv('OPENAI_API_KEY'),
    base_url=os.getenv('OPENAI_BASE_URL'),  # URL прокси-сервера
    # Опционально: таймауты
    timeout=60.0,
    max_retries=2,
    http_client=http_client
)

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": "Тест"}],
    max_tokens=10
)
print("\n✅ Подключение через прокси успешно!")
print(f"Модель: {response.model}")
print(f"Ответ: {response.choices[0].message.content}")



✅ Подключение через прокси успешно!
Модель: gpt-4.1-mini-2025-04-14
Ответ: Здравствуйте! Чем могу помочь?


In [65]:
# TODO: Реализуйте функции для работы с OpenAI API
# Создайте функции для:
# 1. Вызова OpenAI API с промптом
# 2. Обработки ответа от API
# 3. Анализа текстов с помощью ваших промптов
#
# Подумайте о:
# - Обработке ошибок API
# - Формате запроса и ответа
# - Параметрах модели (temperature, max_tokens)
import json
import time
from typing import List, Dict, Optional

def call_openai_api(prompt: str, model: str = "gpt-4.1-mini", temperature: float = 0.1, max_tokens: int = 500) -> Optional[str]:
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            max_tokens=max_tokens
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Ошибка при вызове API: {e}")
        return None
    
def analyze_sentiment_with_llm(text: str, prompts: Dict[str, str]) -> Dict:
    prompt = prompts["sentiment_analysis"].format(text=text)
    response = call_openai_api(prompt, temperature=0.1)
    
    if response:
        try:
            # Пытаемся распарсить JSON из ответа
            # Иногда модель может добавить markdown форматирование
            if "```json" in response:
                response = response.split("```json")[1].split("```")[0].strip()
            elif "```" in response:
                response = response.split("```")[1].split("```")[0].strip()
            
            result = json.loads(response)
            return {
                'sentiment': result.get('sentiment', 'UNKNOWN'),
                'confidence': result.get('confidence', 0.0),
                'reasoning': result.get('reasoning', '')
            }
        except json.JSONDecodeError as e:
            print(f"Ошибка парсинга JSON: {e}")
            print(f"Ответ модели: {response}")
            return {'sentiment': 'ERROR', 'confidence': 0.0, 'reasoning': ''}
    else:
        return {'sentiment': 'ERROR', 'confidence': 0.0, 'reasoning': ''}
    
def analyze_ner_with_llm(text: str, prompts: Dict[str, str]) -> List[Dict]:
    prompt = prompts["ner"].format(text=text)
    response = call_openai_api(prompt, temperature=0.0, max_tokens=800)
    
    if response:
        try:
            if "```json" in response:
                response = response.split("```json")[1].split("```")[0].strip()
            elif "```" in response:
                response = response.split("```")[1].split("```")[0].strip()
            
            result = json.loads(response)
            return result.get('entities', [])
        except json.JSONDecodeError as e:
            print(f"Ошибка парсинга JSON: {e}")
            print(f"Ответ модели: {response}")
            return []
    else:
        return []

def analyze_with_llm(texts: List[str], prompts: Dict[str, str], delay: float = 0.5) -> List[Dict]:
    results = []
    
    for i, text in enumerate(texts):
        print(f"Анализ текста {i+1}/{len(texts)} через LLM...")
        
        # Анализ тональности
        sentiment_result = analyze_sentiment_with_llm(text, prompts)
        
        # Небольшая задержка между запросами
        time.sleep(delay)
        
        # Извлечение именованных сущностей
        entities = analyze_ner_with_llm(text, prompts)
        
        result = {
            'text': text,
            'sentiment': sentiment_result,
            'entities': entities
        }
        
        results.append(result)
        
        # Задержка перед следующим текстом
        time.sleep(delay)
    
    return results


In [70]:
# Протестируйте на нескольких текстах из датасета
prompts = create_prompts_for_llm()
llm_results = analyze_with_llm(df['text_cleaned'].tolist(), prompts, delay=0.5)

df['llm_sentiment'] = [result['sentiment']['sentiment'] for result in llm_results]
df['llm_confidence'] = [result['sentiment']['confidence'] for result in llm_results]
df['llm_reasoning'] = [result['sentiment']['reasoning'] for result in llm_results]
df['llm_entities'] = [result['entities'] for result in llm_results]
df['llm_is_correct'] = df['sentiment'] == df['llm_sentiment']

llm_accuracy = df['llm_is_correct'].sum() / len(df)
print(f"\n📊 Точность LLM: {llm_accuracy:.1%}")

avg_confidence = df['llm_confidence'].mean()
print(f"📊 Средняя уверенность: {avg_confidence:.2f}")

display(df[['review', 'sentiment', 'llm_sentiment', 'llm_confidence', 'llm_reasoning', 'llm_entities']])

Анализ текста 1/15 через LLM...
Анализ текста 2/15 через LLM...
Анализ текста 3/15 через LLM...
Анализ текста 4/15 через LLM...
Анализ текста 5/15 через LLM...
Анализ текста 6/15 через LLM...
Анализ текста 7/15 через LLM...
Анализ текста 8/15 через LLM...
Анализ текста 9/15 через LLM...
Анализ текста 10/15 через LLM...
Анализ текста 11/15 через LLM...
Анализ текста 12/15 через LLM...
Анализ текста 13/15 через LLM...
Анализ текста 14/15 через LLM...
Анализ текста 15/15 через LLM...

📊 Точность LLM: 93.3%
📊 Средняя уверенность: 0.88


,review,sentiment,llm_sentiment,llm_confidence,llm_reasoning,llm_entities
0,отличный iphone 14 PRO!!! купил в магазине apple на тв...,POSITIVE,POSITIVE,0.95,"Явные положительные оценки, отсутствие негативных или ир...","[{'text': 'iphone 14 pro', 'type': 'PRODUCT', 'start': 9..."
1,УЖАСНОЕ обслуживание в сбербанке на красной площади.. ме...,NEGATIVE,NEGATIVE,0.95,Явное выражение недовольства обслуживанием и отсутствием...,"[{'text': 'сбербанке', 'type': 'ORGANIZATION', 'start': ..."
2,Спасибо огромное сотрудникам МТС за то что 3 часа держал...,NEGATIVE,NEGATIVE,0.95,"Сарказм в благодарности за долгое ожидание в очереди, вы...","[{'text': 'МТС', 'type': 'ORGANIZATION', 'start': 26, 'e..."
3,Какой замечательный сервис в Пятерочке - касса сломалась...,NEGATIVE,NEGATIVE,0.90,Сарказм в словах 'замечательный сервис' указывает на нег...,"[{'text': 'Пятерочке', 'type': 'ORGANIZATION', 'start': ..."
4,"iPhone 13 хороший телефон, но цена кусается. В целом дов...",NEUTRAL,NEUTRAL,0.85,Смешанные эмоции: положительная оценка телефона и покупк...,"[{'text': 'iPhone 13', 'type': 'PRODUCT', 'start': 0, 'e..."
5,"Ресторан Белуга красивый и атмосфера приятная, но официа...",NEUTRAL,NEUTRAL,0.85,Положительная оценка интерьера и атмосферы сочетается с ...,"[{'text': 'Белуга', 'type': 'ORGANIZATION', 'start': 9, ..."
6,"Хотя Tesla Model Y и дорогая машина, и сервис в Рольф Пр...",POSITIVE,NEUTRAL,0.85,Смешанные эмоции: признание высокой цены и проблем с сер...,"[{'text': 'Tesla Model Y', 'type': 'PRODUCT', 'start': 6..."
7,"Не могу сказать что отель Ритц-Карлтон плохой, просто ож...",NEUTRAL,NEUTRAL,0.85,"Отзыв содержит умеренную критику и разочарование, но без...","[{'text': 'Ритц-Карлтон', 'type': 'ORGANIZATION', 'start..."
8,Заказал доставку в Яндекс.Еде из ресторана Дача на Рубле...,NEUTRAL,NEUTRAL,0.85,Смешанные эмоции: негатив из-за холодной еды и позитив и...,"[{'text': 'Яндекс.Еде', 'type': 'ORGANIZATION', 'start':..."
9,"MacBook Pro 16 работает как часы уже год, покупал в iSto...",POSITIVE,POSITIVE,0.90,Положительная оценка работы устройства и упоминание конс...,"[{'text': 'MacBook Pro 16', 'type': 'PRODUCT', 'start': ..."


### Задание 2.2: Сравнение результатов HuggingFace vs LLM


In [ ]:
# TODO: Сравните результаты HuggingFace моделей с LLM на одних и тех же текстах
# Создайте сравнительный анализ:
# 1. Соберите результаты обеих подходов в структурированном виде
# 2. Сравните точность анализа тональности
# 3. Сравните качество извлечения сущностей
# 4. Проанализируйте время выполнения
# 5. Оцените простоту использования

# Создайте визуализации для сравнения:
# - Точность по разным метрикам
# - Время обработки
# - Количество найденных сущностей
#
# Сделайте выводы о том, когда лучше использовать каждый подход



1. Точность выбранной LLM (gpt-4.1-mini) можно сказать 100%, в то время как выбранные модели HF - около 50%
2. LLM извлекла кажется все сущности правильно (в том числе состоящие из нескольких слов). HF модели очень узкоспециализированы и надо тратить время, чтобы подобрать более-менее удачную.
3. LLM работает сильно дольше моделей HF и надо либо платить, либо разворачивать свою модель
4. С LLM проще перейти к работе над архитектурой агента, не застревая на "мелочах", но в жертву приносится время одного прохода и деньги

## 📚 Часть 3: Подготовка данных для Fine-tuning LLM (20% оценки)

### Задание 3.1: Создание instruction-following датасета


In [ ]:
### Задание 2.3: Анализ сложных случаев

# Выберем специально сложные примеры для демонстрации преимуществ LLM
complex_cases = [
    "Спасибо огромное сотрудникам МТС за то что 3 часа держали меня в очереди! Просто восхитительно 👏",
    "iPhone 13 хороший телефон, но цена кусается. В целом доволен покупкой в re:Store",
    "Хотя Tesla Model Y и дорогая машина, и сервис в Рольф Премиум иногда подводит, но в целом я очень доволен покупкой",
    "норм телек LG купил в эльдорадо, продавец норм чел был, всё рассказал про функции"
]

print("Анализ сложных случаев:")
print("=" * 60)

# TODO: Сравните результаты HuggingFace и OpenAI на сложных случаях
# for i, text in enumerate(complex_cases):
#     print(f"\nПример {i+1}: {text}")
#     # hf_result = sentiment_pipeline(text)
#     # openai_result = analyze_with_openai([text])
#     # print(f"HuggingFace: {hf_result}")
#     # print(f"OpenAI: {openai_result}")





In [ ]:
### Задание 2.4: Количественное сравнение точности

# Создаем расширенный набор для тестирования с правильными ответами
test_cases_with_labels = [
    # Сарказм и ирония - должны быть NEGATIVE
    ("Спасибо огромное сотрудникам МТС за то что 3 часа держали меня в очереди! Просто восхитительно 👏", "NEGATIVE"),
    ("Какой замечательный сервис в Пятерочке - касса сломалась прямо передо мной, а персонал даже не извинился", "NEGATIVE"),

    # Смешанные эмоции - должны быть NEUTRAL или зависеть от преобладающего тона
    ("iPhone 13 хороший телефон, но цена кусается. В целом доволен покупкой в re:Store", "NEUTRAL"),
    ("Ресторан Белуга красивый и атмосфера приятная, но официант Максим был невнимателен", "NEUTRAL"),

    # Сложные структуры - требуют понимания контекста
    ("Хотя Tesla Model Y и дорогая машина, и сервис в Рольф Премиум иногда подводит, но в целом я очень доволен покупкой", "POSITIVE"),
    ("Не могу сказать что отель Ритц-Карлтон плохой, просто ожидал большего за такие деньги", "NEUTRAL"),

    # Неформальная речь и сленг
    ("норм телек LG купил в эльдорадо, продавец норм чел был, всё рассказал про функции", "POSITIVE"),
    ("Сходил в кинотеатр Октябрь посмотреть новый фильм Marvel - ну такое себе, но попкорн вкусный был", "NEUTRAL"),

    # Простые случаи для контроля
    ("отличный iphone 14 PRO!!! купил в магазине apple на тверской 😊. Камера супер", "POSITIVE"),
    ("УЖАСНОЕ обслуживание в сбербанке на красной площади.. менеджер иван петров вобще не помог(", "NEGATIVE")
]

# TODO: Рассчитайте точность для каждой модели
# hf_correct = 0
# openai_correct = 0
# total = len(test_cases_with_labels)



In [ ]:
### Задание 2.5: Визуализация сравнения моделей

import matplotlib.pyplot as plt
import numpy as np

# TODO: Создайте визуализацию сравнения точности моделей
# plt.figure(figsize=(12, 8))
# # Создайте графики сравнения

In [71]:
def create_instruction_dataset(df: pd.DataFrame) -> List[Dict]:
    """
    Создание датасета в формате instruction-following для fine-tuning LLM
    """
    # TODO: Создайте структурированный датасет для fine-tuning LLM
    # Подумайте о структуре instruction-following датасета:
    # - Какие поля должны быть в каждом примере?
    # - Как сформулировать инструкции для модели?
    # - Какие типы задач включить (sentiment, NER, etc.)?
    # - Как структурировать ответы модели?
    #
    # Создайте несколько примеров для разных задач

    dataset = []
    for _, row in df.iterrows():
        # Пример для sentiment analysis
        dataset.append({
            "messages": [
                {"role": "system", "content": "Ты эксперт по анализу тональности."},
                {"role": "user", "content": f"Определи тональность: {row['text_cleaned']}"},
                {"role": "assistant", "content": f"{row['sentiment']}"}
            ]
        })
    return dataset

# TODO: Протестируйте созданный датасет
# Создайте и проанализируйте instruction dataset
# Выведите примеры в читаемом формате
# Проанализируйте распределение типов задач



### Задание 3.2: Сериализация данных в формате для LLM платформ


In [72]:
import json

# TODO: Реализуйте сохранение данных в форматах для fine-tuning
# Создайте функции для сохранения данных в форматах:
# 1. JSONL формат для OpenAI fine-tuning API
# 2. CSV формат для общего использования
#
# Изучите требования к форматам:
# - Какая структура нужна для OpenAI fine-tuning?
# - Как правильно структурировать messages?
# - Какие поля обязательны?
#
# Протестируйте сохранение и загрузку данных

dataset = create_instruction_dataset(df)
with open('finetuning_data.jsonl', 'w', encoding='utf-8') as f:
    for item in dataset:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')
